# Discovery: Vague Ask to AI Spec

**Phase 11** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-11/03-discovery-vague-ask-to-spec.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You finish a 45-minute discovery call with a customer. You have three pages of notes. You understand the problem intuitively. The stakeholders are aligned, the use case is clear, and you're ready to start building.

Then you open a blank document to write up the spec and realize you don't have a clear success metric. You know what the system should do but not how to measure whether it's doing it. You don't know who signed off on the data access. You're not sure if the output format you're imagining matches what the customer actually expects.

These are not gaps you can fill in later. They are gaps that will surface as blockers at the worst possible moment: during a demo, during handoff, or w...

## The Concept

### From Discovery Notes to AI Spec to Eval Criteria



The spec writer bridges the gap between what the customer said (raw notes) and what the build needs (a testable specification). Claude helps suggest success metrics when the customer only gave a direction, and surfaces unknowns that the notes don't address.

### The 7 Sections of an AI Spec

Every AI spec must have exactly these 7 sections:

```
Section              Purpose                              Blocker if missing
-------------------  -----------------------------------  ----------------------
Problem statement    Describes the pai...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
#!/usr/bin/env python3
"""
AI Spec Writer

Takes rough discovery notes and uses Claude to structure them into
a 7-section AI spec. Flags missing or unconfirmed sections.

Usage:
    python main.py --notes notes.txt
    python main.py --notes notes.txt --output spec.json
    python main.py --interactive
"""
import json
import sys
import argparse
import re
from pathlib import Path
import anthropic

MODEL = "claude-3-5-haiku-20241022"

SPEC_SYSTEM_PROMPT = """You are an AI spec writer for a Forward-Deployed Engineering team.
Your job is to take rough discovery notes and structure them into a formal AI spec.

The AI spec has exactly 7 sections. Output each section on its own line starting with
the section number and name in all-caps, followed by the content.

Sections:
1. PROBLEM STATEMENT - Describe the pain in customer terms, not engineering terms.
2. SUCCESS METRIC - Must include: number, baseline, time horizon, measurement source.
   If notes don't contain a specific metric, suggest one and prefix with [SUGGESTED].
3. INPUT/OUTPUT CONTRACT - What goes in, what comes out, in what format.
4. DATA SOURCES - Owner (named person), system, format, access path.
   If unknown, write [UNKNOWN].
5. INTEGRATION POINTS - Where output lands, who acts on it, error handling.
6. RISKS AND UNKNOWNS - Unresolved questions that could affect the build.
7. OUT OF SCOPE - What this system will NOT do in version 1.

Rules:
- Do not invent facts. Only use information present in the notes.
- If a section has no information in the notes, write [UNKNOWN - must resolve before build].
- If you suggest a success metric, it must be plausible from the notes and flagged [SUGGESTED].
- Keep each section concise: 1-4 bullet points or sentences.
- End with a line: FLAGS: followed by any [SUGGESTED] or [UNKNOWN] items that must be resolved.
"""

SPEC_USER_TEMPLATE = """Here are the raw discovery notes:

---
{notes}
---

Please write the AI spec."""

SECTION_NAMES = [
    "PROBLEM STATEMENT",
    "SUCCESS METRIC",
    "INPUT/OUTPUT CONTRACT",
    "DATA SOURCES",
    "INTEGRATION POINTS",
    "RISKS AND UNKNOWNS",
    "OUT OF SCOPE",
]


def parse_spec(raw: str) -> dict:
    """Parse Claude's spec output into sections."""
    sections: dict[str, str] = {}
    flags: list[str] = []

    current_section = None
    current_lines: list[str] = []

    for line in raw.splitlines():
        stripped = line.strip()

        # Check if this line starts a new section
        matched_section = None
        for name in SECTION_NAMES:
            pattern = rf"^\d+\.\s+{re.escape(name)}"
            if re.match(pattern, stripped, re.IGNORECASE):
                matched_section = name
                break

        if matched_section:
            if current_section:
                sections[current_section] = "\n".join(current_lines).strip()
            current_section = matched_section
            # Capture text on the same line after the section heading
            rest = re.sub(rf"^\d+\.\s+{re.escape(matched_section)}\s*[-:]?\s*", "", stripped, flags=re.IGNORECASE)
            current_lines = [rest] if rest else []

        elif stripped.upper().startswith("FLAGS:"):
            if current_section:
                sections[current_section] = "\n".join(current_lines).strip()
                current_section = None
            flag_text = stripped[6:].strip()
            if flag_text:
                flags.append(flag_text)

        else:
            if current_section:
                current_lines.append(stripped)
            elif stripped.startswith("-") or stripped.startswith("*"):
                flags.append(stripped.lstrip("-* "))

    if current_section:
        sections[current_section] = "\n".join(current_lines).strip()

    # Collect inline flags from section content
    for name, content in sections.items():
        if "[SUGGESTED]" in content or "[UNKNOWN]" in content:
            flags.append(f"Section '{name}' has unresolved items.")

    return {"sections": sections, "flags": list(dict.fromkeys(flags))}


def call_claude(notes: str, client: anthropic.Anthropic) -> str:
    """Call Claude to generate the AI spec."""
    message = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system=SPEC_SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": SPEC_USER_TEMPLATE.format(notes=notes)}
        ],
    )
    return message.content[0].text


def print_spec(parsed: dict) -> None:
    print("\n" + "=" * 55)
    print("AI SPEC")
    print("=" * 55)

    for i, name in enumerate(SECTION_NAMES, 1):
        content = parsed["sections"].get(name, "[UNKNOWN - must resolve before build]")
        print(f"\n{i}. {name}")
        for line in content.splitlines():
            if line.strip():
                print(f"   {line}")

    flags = parsed.get("flags", [])
    if flags:
        print("\n" + "=" * 55)
        print(f"FLAGS ({len(flags)} items to resolve before build)")
        print("=" * 55)
        for flag in flags:
            print(f"  * {flag}")
    else:
        print("\nNo flags. Spec is complete.")


def collect_notes_interactive() -> str:
    print("\n=== AI SPEC WRITER ===")
    print("Paste your discovery notes below.")
    print("Enter a blank line followed by 'END' when done.\n")
    lines = []
    while True:
        try:
            line = input()
        except EOFError:
            break
        if line.strip().upper() == "END":
            break
        lines.append(line)
    return "\n".join(lines)


def main() -> None:
    parser = argparse.ArgumentParser(description="AI Spec Writer")
    parser.add_argument("--notes", metavar="FILE", help="Path to discovery notes file")
    parser.add_argument("--output", metavar="FILE", help="Export spec to JSON file")
    parser.add_argument("--markdown", metavar="FILE", help="Export spec to Markdown file")
    parser.add_argument("--interactive", action="store_true", help="Enter notes interactively")
    args = parser.parse_args()

    if args.notes:
        notes_path = Path(args.notes)
        if not notes_path.exists():
            print(f"Error: file not found: {args.notes}", file=sys.stderr)
            sys.exit(1)
        notes = notes_path.read_text()
    elif args.interactive:
        notes = collect_notes_interactive()
    else:
        parser.print_help()
        sys.exit(0)

    if not notes.strip():
        print("Error: notes are empty.", file=sys.stderr)
        sys.exit(1)

    print("\nGenerating AI spec from discovery notes...")
    client = anthropic.Anthropic()
    raw = call_claude(notes, client)
    parsed = parse_spec(raw)

    print_spec(parsed)

    if args.output:
        with open(args.output, "w") as f:
            json.dump(
                {
                    "sections": parsed["sections"],
                    "flags": parsed["flags"],
                    "raw": raw,
                },
                f,
                indent=2,
            )
        print(f"\nSpec exported to {args.output}")

    if args.markdown:
        lines = ["# AI Spec\n"]
        for i, name in enumerate(SECTION_NAMES, 1):
            content = parsed["sections"].get(name, "[UNKNOWN - must resolve before build]")
            lines.append(f"## {i}. {name}\n")
            lines.append(content + "\n")
        if parsed["flags"]:
            lines.append("## Flags\n")
            for flag in parsed["flags"]:
                lines.append(f"- {flag}")
        with open(args.markdown, "w") as f:
            f.write("\n".join(lines))
        print(f"Markdown spec exported to {args.markdown}")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What is the correct next action?**

- A. Accept the suggested metric and start building since it is reasonable
- B. Send the customer a single confirmation question: 'Does this metric match your expectation?'
- C. Remove the metric and build first, then define success based on what you observe
- D. Replace the suggested metric with a broader goal to avoid being held to a specific number

**2. What is the risk of starting the build with this flag unresolved?**

- A. Minor risk: you can always add the data connection later in the build
- B. Low risk: Salesforce has a well-documented API so access should be straightforward
- C. High risk: you may build a system whose primary data source is inaccessible, requiring a rebuild
- D. Medium risk: you can use synthetic data while waiting for access confirmation

**3. What is wrong with this success metric?**

- A. Nothing: the customer confirmed it, so it is a valid success criterion
- B. It lacks a number, baseline, time horizon, and measurement source, making it unevaluable
- C. It is too technical; it should use simpler language
- D. It should include the name of the AI model to be used

**4. What should you do in the 3 days before the follow-up?**

- A. Start building toward the most likely integration point to save time
- B. Build the core AI logic without the integration layer, so you can add it once confirmed
- C. Email one specific question: 'Which tool or screen should the AI output appear in?'
- D. Ask a colleague who worked with this customer type before to guess the integration point

**5. How should you respond?**

- A. Accept it: a blank risk section means the spec is clean
- B. Ask them to fill in at least the standard risks: model latency, data format variance, and integration constraints
- C. Add your own risks to their spec without asking
- D. Tell them to start building and document risks as they appear

**6. How does the out-of-scope section of the spec help you handle this?**

- A. It doesn't: the VP outranks the original stakeholder, so the spec must be updated
- B. It provides a documented baseline to reference, so you can say: 'Here is what was scoped. Adding these features changes the timeline. Shall we do a scoping session for the expanded ask?'
- C. You can use the spec to reject the new requirements permanently
- D. The out-of-scope section should be updated immediately to include the new requirements

**7. What does this outcome reveal about the discovery process?**

- A. The spec was correct: the system met its defined goal
- B. The eval was too narrow: it should have tested response drafting too
- C. The discovery call missed a stakeholder expectation that was never captured in the spec
- D. The customer changed their requirements after sign-off, which is not a discovery failure

_Answers are in `checks.json` in the lesson directory._